# GEPA: оптимизация промпта для русской орфографии

Вместо SFT (дообучение весов) здесь используется [GEPA](https://github.com/gepa-ai/gepa) (Genetic-Pareto) — эволюционный оптимизатор **промптов**. Базовая модель **Qwen3.5-0.8B** остаётся замороженной; GEPA ищет лучшую system-инструкцию на датасете [`BW/RU_SPELLCHECK_DEVICE`](https://huggingface.co/datasets/BW/RU_SPELLCHECK_DEVICE).

**Нужно:** GPU (Colab T4+) для Qwen, `API_KEY` для reflection LM ([Cloud.ru Foundation Models](https://foundation-models.api.cloud.ru/v1)).

**Runtime → Run all**

## 1. Авторизация и API-ключи

In [1]:
import os
from huggingface_hub import login

API_BASE_URL = "https://foundation-models.api.cloud.ru/v1"
REFLECTION_MODEL = "openai/gpt-oss-120b"

# Colab: можно использовать секреты
# from google.colab import userdata
# hf_token = userdata.get('HF_TOKEN')
# api_key = userdata.get('API_KEY')

hf_token = 'fake'#os.environ.get('HF_TOKEN', '')
api_key = 'fake'#os.environ.get('API_KEY', '')

if hf_token:
    login(hf_token)
if not api_key:
    print('⚠️  API_KEY не задан — reflection LM не сможет работать')


## 2. Установка зависимостей

In [2]:
%%capture
import os, re

!pip install gepa openai

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec

import torch
torch._dynamo.config.recompile_limit = 64


## 3. Загрузка базовой модели Qwen3.5-0.8B

In [4]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

MODEL_NAME = "unsloth/Qwen3.5-0.8B"
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    full_finetuning=False,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen3-instruct",
)


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

==((====))==  Unsloth 2026.7.2: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

## 4. Подготовка данных для GEPA

In [5]:
from datasets import load_dataset
from gepa.adapters.default_adapter.default_adapter import DefaultDataInst

DATASET_NAME = "BW/RU_SPELLCHECK_DEVICE"
dataset = load_dataset(DATASET_NAME)


def to_gepa_examples(split) -> list[DefaultDataInst]:
    examples = []
    for row in split:
        examples.append({
            "input": row["typed"],
            "answer": row["original"],
            "additional_context": {
                "device_type": row.get("device_type", "unknown"),
            },
        })
    return examples


trainset = to_gepa_examples(dataset["train"])
valset = to_gepa_examples(dataset["test"])
testset = valset  # официальный test split

print(f"Train: {len(trainset)}, Val/Test: {len(valset)}")
print(f"Пример input:  {trainset[0]['input']}")
print(f"Пример answer: {trainset[0]['answer']}")


README.md:   0%|          | 0.00/598 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/69.4k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/67.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/296 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/296 [00:00<?, ? examples/s]

Train: 296, Val/Test: 296
Пример input:  Две молодые демушки вели ее под рукию
Пример answer: Две молодые девушки вели ее под руки.


## 5. Метрики и evaluator для GEPA

In [6]:
import difflib
import re

from gepa.adapters.default_adapter.default_adapter import EvaluationResult


def tokenize_spellcheck(text: str) -> list[str]:
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def extract_edits(source: str, target: str) -> dict[tuple[int, int], tuple[str, ...]]:
    src_tokens = tokenize_spellcheck(source)
    tgt_tokens = tokenize_spellcheck(target)
    edits: dict[tuple[int, int], tuple[str, ...]] = {}
    matcher = difflib.SequenceMatcher(None, src_tokens, tgt_tokens)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            continue
        replacement = tuple(tgt_tokens[j1:j2])
        if src_tokens[i1:i2] != list(replacement):
            edits[(i1, i2)] = replacement
    return edits


def edit_f1(source: str, correction: str, prediction: str) -> dict[str, float]:
    gold = extract_edits(source, correction)
    pred = extract_edits(source, prediction)
    tp = sum(1 for key, val in pred.items() if gold.get(key) == val)
    n_pred, n_gold = len(pred), len(gold)
    precision = tp / n_pred if n_pred else 1.0
    recall = tp / n_gold if n_gold else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "num_gold": n_gold,
        "num_pred": n_pred,
    }


def format_edits(edits: dict[tuple[int, int], tuple[str, ...]], src_tokens: list[str]) -> str:
    if not edits:
        return "(нет правок)"
    lines = []
    for (i, j), repl in sorted(edits.items()):
        src_part = " ".join(src_tokens[i:j])
        tgt_part = " ".join(repl)
        lines.append(f"  '{src_part}' → '{tgt_part}'")
    return "\n".join(lines)


class SpellcheckEditF1Evaluator:
    """Edit-level F1 как score + подробный feedback для reflection LM."""

    def __call__(self, data, response: str) -> EvaluationResult:
        source = data["input"]
        correction = data["answer"]
        prediction = response.strip()

        metrics = edit_f1(source, correction, prediction)
        score = metrics["f1"]

        src_tokens = tokenize_spellcheck(source)
        gold_edits = extract_edits(source, correction)
        pred_edits = extract_edits(source, prediction)

        missed = {k: v for k, v in gold_edits.items() if pred_edits.get(k) != v}
        extra = {k: v for k, v in pred_edits.items() if gold_edits.get(k) != v}

        feedback_parts = [
            f"Исходный текст: {source}",
            f"Эталон: {correction}",
            f"Предсказание: {prediction}",
            f"Edit F1: {metrics['f1']:.3f} (P={metrics['precision']:.3f}, R={metrics['recall']:.3f})",
            f"Устройство ввода: {data['additional_context'].get('device_type', 'unknown')}",
        ]

        if prediction == correction:
            feedback_parts.append("✓ Точное совпадение с эталоном.")
        else:
            if missed:
                feedback_parts.append("Пропущенные правки (нужно исправить, но модель не исправила):")
                feedback_parts.append(format_edits(missed, src_tokens))
            if extra:
                feedback_parts.append("Лишние/неверные правки (модель изменила лишнее):")
                feedback_parts.append(format_edits(extra, src_tokens))
            if len(prediction) > len(correction) * 1.3:
                feedback_parts.append("Проблема: ответ слишком длинный — возможно, модель перефразирует вместо минимальной правки.")

        return EvaluationResult(score=score, feedback="\n".join(feedback_parts))


## 6. Обёртка Qwen для GEPA DefaultAdapter

In [7]:
class QwenChatModel:
    """ChatCompletionCallable для локальной Qwen через Unsloth."""

    def __init__(self, model, tokenizer, max_new_tokens: int = 256):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens

    def __call__(self, messages: list[dict]) -> str:
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = self.tokenizer(
            text=[text],
            images=None,
            videos=None,
            return_tensors="pt",
        ).to(self.model.device)

        input_len = inputs["input_ids"].shape[1]
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            do_sample=False,
        )
        return self.tokenizer.decode(
            outputs[0][input_len:],
            skip_special_tokens=True,
        ).strip()


task_lm = QwenChatModel(model, tokenizer)
evaluator = SpellcheckEditF1Evaluator()


## 7. Запуск GEPA

In [ ]:
import gepa
from openai import OpenAI

SEED_PROMPT = (
    '''
    You are a Russian‑language proofreader.  
    For every input you receive, perform **only** the following actions:

    1. **Correct** all orthographic (spelling), punctuation, and register/case mistakes.  
    - Fix misspelled words, missing or extra letters, and wrong word forms.  
    - Use proper Russian punctuation: commas, periods, question/exclamation marks, em‑dashes (—) for dialogue, hyphens (‑) only where a hyphen is required, and matching quotation marks.  
    - Ensure correct capitalization: the first word of a sentence (or after an em‑dash that starts a new sentence) must start with a capital letter; proper nouns must be capitalized; other words should be lower‑case unless the context demands otherwise.  
    - Preserve the original meaning; do not add, delete, or replace words unless it is necessary to fix an error.

    2. **Output** **exactly** the corrected text, **nothing else**:
    - No explanations, reasoning, or meta‑information.  
    - No tags such as `<think>` or any other markup.  
    - Do **not** surround the result with quotation marks.  
    - Keep the original line breaks (if any) and spacing, adjusting only what is required for correct punctuation (e.g., a single space after a comma, period, dash, etc.).  
    - If the input is already correct, output it unchanged.

    **Examples**

    - Input: `Дают приятрыз дух, увеселяют взор И вам якрасавицы, хранят чебя в цбор.`  
    Output: `Дают приятный дух, увеселяют взор, и вам, красавицы, хранят себя в убор.`

    - Input: `- Чей эоо дом? - спросил он у будувого будочника.`  
    Output: `— Чей это дом? — спросил он у углового будочника.`

    - Input: `Тихий ветер шевелмл листья старого дерева.`  
    Output: `Тихий ветер шевелил листья старого дерева.`

    Follow these rules for every request.
    '''
)

seed_candidate = {"system_prompt": SEED_PROMPT}
# valset = valset[:30]     # уменьшить val
# trainset = trainset[:50]
REFLECTION_LM = REFLECTION_MODEL
reflection_client = OpenAI(api_key=api_key, base_url=API_BASE_URL)


def reflection_lm(prompt: str | list[dict]) -> str:
    """Cloud.ru через OpenAI SDK (как в test_api.py; litellm даёт 404 на этом endpoint)."""
    messages = (
        [{"role": "user", "content": prompt}]
        if isinstance(prompt, str)
        else prompt
    )
    response = reflection_client.chat.completions.create(
        model=REFLECTION_MODEL,
        messages=messages,
        max_tokens=2500,
        temperature=0.5,
        presence_penalty=0,
        top_p=0.95,
    )
    return response.choices[0].message.content

MAX_METRIC_CALLS = 2000  # бюджет оценок; увеличьте для лучшего результата
RUN_DIR = "gepa_spellcheck_run"  # перед новым запуском удалите папку, иначе GEPA продолжит старый run

result = gepa.optimize(
    seed_candidate=seed_candidate,
    trainset=trainset,
    valset=valset,
    task_lm=task_lm,
    evaluator=evaluator,
    reflection_lm=reflection_lm,
    max_metric_calls=MAX_METRIC_CALLS,
    reflection_minibatch_size=5,
    run_dir=RUN_DIR,
    display_progress_bar=True,
    seed=3407,
)

print("\n=== Лучший промпт ===")
print(result.best_candidate["system_prompt"])
print(f"\nЛучший val score (avg): {result.val_aggregate_scores[result.best_idx]:.4f}")
print(f"Всего кандидатов: {result.num_candidates}, metric calls: {result.total_metric_calls}")


GEPA Optimization:   0%|          | 0/2000 [00:00<?, ?rollouts/s]Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not

Iteration 0: Base program full valset score: 0.10629897504897505 over 296 / 296 examples
Iteration 1: Selected program 0 score: 0.10629897504897505


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Iteration 1: Proposed new text for system_prompt: **You are a Russian‑language proofreader.**  
For every piece of text you receive, perform **only** the actions listed below and output **exactly** the corrected text—nothing else.

---

### 1️⃣ What you may change  

| Category | What to fix | How to fix it |
|----------|-------------|---------------|
| **Spelling** | Any misspelled word, missing or extra letters, wrong word form. | Replace the word with its correct spelling. Do **not** change the word’s meaning or replace it with a synonym. |
| **Punctuation** | Missing commas, periods, question/exclamation marks, em‑dashes, hyphens, quotation marks, etc. | Insert, delete or move punctuation so that the sentence follows standard Russian rules. Keep the original order of words. |
| **Capitalisation** | First word of a sentence, first word after an em‑dash that starts a new sentence, proper nouns, acronyms, etc. | Adjust the case of the first letter only where required. |
| **Register /

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
GEPA Optimization:  15%|#5        | 306/2000 [14:23<1:23:06,  2.94s/rollouts]Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


...

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Iteration 13: Proposed new text for system_prompt: **You are a Russian‑language proofreader.**  
For every piece of text you receive you must perform **only** the actions listed below and output **exactly** the corrected text – nothing else.

---

### 1. What you may change  

| Category | What to fix | How to fix it |
|----------|-------------|---------------|
| **Spelling / orthography** | Any misspelled word, extra or missing letters, wrong word form, wrong register or case (e.g. *сердино → сердито*) | Replace the word with the correct form. Do **not** replace it with a synonym or re‑phrase the sentence. |
| **Punctuation** | • Missing or superfluous commas, periods, question/exclamation marks, colons, semicolons, quotation marks.<br>• Hyphens used as dialogue dashes.<br>• Wrong dash type (‑ vs —).<br>• Missing colon before direct speech after introductory words (e.g. *наконец —* → *наконец: —*). | • Use standard Russian punctuation. <br>• Replace every hyphen that introduces or sep

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
GEPA Optimization:  36%|###6      | 722/2000 [1:10:10<2:32:51,  7.18s/rollouts]Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Iteration 13: New subsample score 0.28571428571428575 is not better than old score 0.6190476190476191, skipping
Iteration 14: Selected program 0 score: 0.10629897504897505


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


## 8. Сохранение результата

In [10]:
import json
from pathlib import Path

from gepa.adapters.default_adapter.default_adapter import DefaultAdapter


def evaluate_prompt_on_testset(prompt: str, examples: list) -> dict:
    adapter = DefaultAdapter(model=task_lm, evaluator=evaluator)
    candidate = {"system_prompt": prompt}
    batch = adapter.evaluate(examples, candidate, capture_traces=False)

    per_example = []
    exact = 0
    for ex, out, score in zip(examples, batch.outputs, batch.scores):
        prediction = out["full_assistant_response"].strip()
        gold = ex["answer"].strip()
        is_exact = prediction == gold
        exact += int(is_exact)
        per_example.append({
            "input": ex["input"],
            "gold": ex["answer"],
            "prediction": prediction,
            "edit_f1": round(score, 4),
            "exact_match": is_exact,
            "device_type": ex["additional_context"].get("device_type", "unknown"),
        })

    total = len(examples)
    return {
        "aggregate": {
            "avg_edit_f1": round(sum(batch.scores) / total, 4),
            "exact_match": round(exact / total, 4),
            "exact_match_count": exact,
            "total": total,
        },
        "examples": per_example,
    }


print("Оценка на test до GEPA (seed prompt)...")
test_before_gepa = evaluate_prompt_on_testset(SEED_PROMPT, testset)

print("Оценка на test после GEPA (optimized prompt)...")
test_after_gepa = evaluate_prompt_on_testset(result.best_candidate["system_prompt"], testset)

OUTPUT_PATH = Path("gepa_spellcheck_prompt.json")
payload = {
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "test_split": "test",
    "seed_prompt": SEED_PROMPT,
    "optimized_prompt": result.best_candidate["system_prompt"],
    "val_scores": result.val_aggregate_scores,
    "max_metric_calls": MAX_METRIC_CALLS,
    "reflection_lm": REFLECTION_LM,
    "reflection_api_base": API_BASE_URL,
    "test_evaluation": {
        "before_gepa": test_before_gepa,
        "after_gepa": test_after_gepa,
        "delta": {
            "avg_edit_f1": round(
                test_after_gepa["aggregate"]["avg_edit_f1"]
                - test_before_gepa["aggregate"]["avg_edit_f1"],
                4,
            ),
            "exact_match": round(
                test_after_gepa["aggregate"]["exact_match"]
                - test_before_gepa["aggregate"]["exact_match"],
                4,
            ),
        },
    },
}
OUTPUT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Сохранено: {OUTPUT_PATH.resolve()}")
print("Test до GEPA:", test_before_gepa["aggregate"])
print("Test после GEPA:", test_after_gepa["aggregate"])


Сохранено: /content/gepa_spellcheck_prompt.json


## 9. Оценка seed vs optimized на test

In [ ]:
print("Seed prompt (test):")
print(test_before_gepa["aggregate"])
print("\nOptimized prompt (test):")
print(test_after_gepa["aggregate"])
print("\nDelta:")
print(payload["test_evaluation"]["delta"])


## 10. Примеры предсказаний

In [ ]:
adapter = DefaultAdapter(model=task_lm, evaluator=evaluator)
optimized_candidate = {"system_prompt": result.best_candidate["system_prompt"]}
demo_batch = adapter.evaluate(testset[:5], optimized_candidate, capture_traces=False)

for ex, out, score in zip(testset[:5], demo_batch.outputs, demo_batch.scores):
    print("---")
    print(f"typed:    {ex['input']}")
    print(f"gold:     {ex['answer']}")
    print(f"pred:     {out['full_assistant_response']}")
    print(f"edit F1:  {score:.3f}")
